In [3]:

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "MIG-e6b94c71-1db7-5ad2-a95a-9de8bd50bb1a"


import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import h5py
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score
from sklearn.utils.class_weight import compute_class_weight
import time
from tqdm.auto import tqdm 
LEARNING_RATE = 0.0005

BATCH_SIZE = 32 
NUM_EPOCHS = 30 
NUM_CLASSES = 12

TRAIN_DATA_FILE = 'train.h5'
TEST_DATA_FILE = 'test.h5'
TEST_LABELS_FILE = 'test_labels.csv' 

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def load_and_preprocess_data():
    """Loads, concatenates, transposes, and scales all data."""
    
    print("Loading labels...")
    with h5py.File(TRAIN_DATA_FILE, 'r') as hdf:
        y_train = np.array(hdf.get('Class/1'))

    y_test = pd.read_csv(TEST_LABELS_FILE, header=None).values.squeeze()

    print("Loading and concatenating training spectra...")
    all_train_spectra = []
    with h5py.File(TRAIN_DATA_FILE, 'r') as hdf:
        train_keys = [f"{i:03d}" for i in range(1, 101)]
        for key in tqdm(train_keys, desc="Training data"):
            data_block = np.array(hdf[f'Spectra/{key}'])
            all_train_spectra.append(data_block.T)
    
    X_train_raw = np.concatenate(all_train_spectra, axis=0)

    print("Loading and concatenating test spectra...")
    all_test_spectra = []
    with h5py.File(TEST_DATA_FILE, 'r') as hdf:
        test_keys = ['1', '2']
        for key in tqdm(test_keys, desc="Test data    "):
            data_block = np.array(hdf[f'UNKNOWN/{key}'])
            all_test_spectra.append(data_block.T)
            
    X_test_raw = np.concatenate(all_test_spectra, axis=0)

    print(f"Raw X_train shape: {X_train_raw.shape}, Raw y_train shape: {y_train.shape}")
    print(f"Raw X_test shape: {X_test_raw.shape}, Raw y_test shape: {y_test.shape}")

    print("Scaling data (this can take several minutes)...")
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_raw)
    X_test_scaled = scaler.transform(X_test_raw)
    
    print("Data loading and preprocessing complete.")
    return X_train_scaled, y_train, X_test_scaled, y_test



class LibsDataset(Dataset):
    """Custom PyTorch Dataset for LIBS spectra (for CNN input)."""
    def __init__(self, data, labels):
        # CNN expects: (N, Channels, Length)
        self.X = torch.tensor(data, dtype=torch.float32).unsqueeze(1)
        self.y = torch.tensor(labels, dtype=torch.long)
        
        # Labels are 1-indexed (1-12), convert to 0-indexed (0-11)
        self.y = self.y - 1 

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]



class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1, downsample=None):
        super(ResidualBlock, self).__init__()
        
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size=kernel_size, 
                               stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm1d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.dropout1 = nn.Dropout(0.2) # Regularization
        
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size=kernel_size, 
                               stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm1d(out_channels)
        self.dropout2 = nn.Dropout(0.2) # Regularization
        
        self.downsample = downsample

    def forward(self, x):
        identity = x # Save the skip connection
        
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        out = self.dropout1(out)
        
        out = self.conv2(out)
        out = self.bn2(out)
        
        if self.downsample is not None:
            identity = self.downsample(x) # Apply downsampling to the skip connection if needed

        out += identity # Add the skip connection back
        out = self.relu(out)
        out = self.dropout2(out)
        
        return out


class ResNet1D(nn.Module):
    def __init__(self, block, layers, num_classes=12, in_channels=1):
        super(ResNet1D, self).__init__()
        
        self.in_channels = 64
        
        # Initial Convolution
        self.conv1 = nn.Conv1d(in_channels, self.in_channels, kernel_size=7, 
                               stride=2, padding=3, bias=False)
        self.bn1 = nn.BatchNorm1d(self.in_channels)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool1d(kernel_size=3, stride=2, padding=1)
        
        # ResNet Layers
        self.layer1 = self._make_layer(block, 64, layers[0], stride=1)
        self.layer2 = self._make_layer(block, 128, layers[1], stride=2)
        self.layer3 = self._make_layer(block, 256, layers[2], stride=2)
        self.layer4 = self._make_layer(block, 512, layers[3], stride=2)
        
        # Final Classifier
        self.avgpool = nn.AdaptiveAvgPool1d(1) # Smart pooling layer
        self.flatten = nn.Flatten()
        self.fc = nn.Linear(512, num_classes)

    def _make_layer(self, block, out_channels, blocks, stride=1):
        downsample = None
        if stride != 1 or self.in_channels != out_channels:
            downsample = nn.Sequential(
                nn.Conv1d(self.in_channels, out_channels, kernel_size=1, 
                          stride=stride, bias=False),
                nn.BatchNorm1d(out_channels),
            )

        layers = []
        layers.append(block(self.in_channels, out_channels, stride=stride, downsample=downsample))
        self.in_channels = out_channels
        for _ in range(1, blocks):
            layers.append(block(self.in_channels, out_channels))

        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x)
        x = self.flatten(x)
        x = self.fc(x)
        return x


def train_model(model, train_loader, criterion, optimizer, epoch):
    model.train()
    total_loss = 0
    start_time = time.time()
    
    for i, (spectra, labels) in enumerate(tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}")):
        spectra = spectra.to(device)
        labels = labels.to(device)
        
        outputs = model(spectra)
        loss = criterion(outputs, labels)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
            
    avg_loss = total_loss / len(train_loader)
    epoch_time = time.time() - start_time
    print(f'--- Epoch {epoch+1} Summary ---')
    print(f'Average Training Loss: {avg_loss:.4f}, Time: {epoch_time:.2f}s')



def evaluate_model(model, test_loader, criterion):
    print("\nEvaluating model on test data...")
    model.eval()
    
    all_preds = []
    all_labels = []
    total_loss = 0
    
    with torch.no_grad():
        for spectra, labels in tqdm(test_loader, desc="Evaluating"):
            spectra = spectra.to(device)
            labels = labels.to(device)
            
            outputs = model(spectra)
            loss = criterion(outputs, labels)
            total_loss += loss.item()
            
            _, predicted = torch.max(outputs.data, 1)
            
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(test_loader)
    accuracy = accuracy_score(all_labels, all_preds)
    
    print(f'Test Loss: {avg_loss:.4f}')
    print(f'Test Accuracy: {accuracy * 100:.2f}%')
    
    print("\n--- Classification Report ---")
    target_names = [f"Class {i+1}" for i in range(NUM_CLASSES)]
    print(classification_report(all_labels, all_preds, labels=list(range(NUM_CLASSES)), target_names=target_names, digits=4, zero_division=0))



if __name__ == "__main__":
    
    print(f"Using device: {device}")
    
    # --- Data ---
    X_train, y_train, X_test, y_test = load_and_preprocess_data()
    
    print("\nCalculating class weights for weighted loss...")
    class_weights = compute_class_weight(
        'balanced',
        classes=np.unique(y_train),
        y=y_train
    )
    class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)
    print(f"Weights calculated: {class_weights_tensor}")

    print("\nCreating PyTorch Datasets and DataLoaders...")
    train_dataset = LibsDataset(X_train, y_train)
    test_dataset = LibsDataset(X_test, y_test)
    

    
    train_loader = DataLoader(dataset=train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
    test_loader = DataLoader(dataset=test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

    print(f"Total training samples: {len(train_dataset)}")
    print(f"Total test samples: {len(test_dataset)}")

    # --- Model ---
   # ResNet-18 has 4 layers with [2, 2, 2, 2] blocks each
    model = ResNet1D(ResidualBlock, [2, 2, 2, 2], num_classes=NUM_CLASSES).to(device)
    print("\nNEW ResNet-18 (V3) Model architecture created and moved to GPU.")
    print(f"Total parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

    """ --- Loss, Optimizer, Scheduler ---"""
    criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=15, gamma=0.5)
    print("Using Weighted Loss, Adam Optimizer, and StepLR Scheduler.")

    # --- Training ---
    print(f"\nStarting training for {NUM_EPOCHS} epochs...")
    total_start_time = time.time()
    
    for epoch in range(NUM_EPOCHS):
        train_model(model, train_loader, criterion, optimizer, epoch)
        scheduler.step()
        current_lr = optimizer.param_groups[0]['lr']
        print(f"--- End of Epoch {epoch+1}, Current LR: {current_lr} ---")
        
    total_end_time = time.time()
    print(f"\n--- Training Finished ---")
    print(f"Total training time: {(total_end_time - total_start_time)/60:.2f} minutes")

    # --- Evaluation ---
    evaluate_model(model, test_loader, criterion)

Using device: cuda
Loading labels...
Loading and concatenating training spectra...


Training data:   0%|          | 0/100 [00:00<?, ?it/s]

Loading and concatenating test spectra...


Test data    :   0%|          | 0/2 [00:00<?, ?it/s]

Raw X_train shape: (50000, 40002), Raw y_train shape: (50000,)
Raw X_test shape: (20000, 40002), Raw y_test shape: (20000,)
Scaling data (this can take several minutes)...
Data loading and preprocessing complete.

Calculating class weights for weighted loss...
Weights calculated: tensor([0.9259, 0.5556, 1.6667, 1.3889, 1.1905, 0.9259, 1.6667, 1.3889, 0.5556,
        1.6667, 0.6944, 1.3889], device='cuda:0')

Creating PyTorch Datasets and DataLoaders...
Total training samples: 50000
Total test samples: 20000

NEW ResNet-18 (V3) Model architecture created and moved to GPU.
Total parameters: 3,850,060
Using Weighted Loss, Adam Optimizer, and StepLR Scheduler.

Starting training for 30 epochs...


...

--- Epoch 29 Summary ---
Average Training Loss: 0.0298, Time: 248.69s
--- End of Epoch 29, Current LR: 0.00025 ---


Epoch 30/30:   0%|          | 0/1563 [00:00<?, ?it/s]

--- Epoch 30 Summary ---
Average Training Loss: 0.0287, Time: 245.05s
--- End of Epoch 30, Current LR: 0.000125 ---

--- Training Finished ---
Total training time: 153.69 minutes

Evaluating model on test data...


Evaluating:   0%|          | 0/625 [00:00<?, ?it/s]

Test Loss: 3.2745
Test Accuracy: 67.40%

--- Classification Report ---
              precision    recall  f1-score   support

     Class 1     0.6980    0.4504    0.5475      3146
     Class 2     0.9700    0.8993    0.9333      3129
     Class 3     0.8182    0.0829    0.1505       543
     Class 4     0.7996    0.7143    0.7545      1603
     Class 5     0.2752    0.6393    0.3847      1048
     Class 6     0.5016    0.9994    0.6679      1589
     Class 7     0.1243    0.1571    0.1388       560
     Class 8     0.8809    0.9185    0.8993      1546
     Class 9     0.7807    0.8858    0.8300      3127
    Class 10     0.4798    0.9494    0.6375       514
    Class 11     0.9495    0.3236    0.4827      3195
    Class 12     0.0000    0.0000    0.0000         0

    accuracy                         0.6740     20000
   macro avg     0.6065    0.5850    0.5356     20000
weighted avg     0.7598    0.6740    0.6666     20000



In [1]:
import torch
print(f"Is CUDA available in this notebook? {torch.cuda.is_available()}")

Is CUDA available in this notebook? True
